In [9]:
%%file producer.py 
#jak odpale komorke to ona sie nie uruchomi, kod zapisze się do pliku producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8') #serializacja, zeby broker (napisany w javie) odbieral slowniki napisane w pythonie
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    #slownik, jakbysmy chcieli w projekcie uzyc danych np. z csv to musielibysmy je zamienic na slownik, przed przesylaniem ich strumieniowo
    czas = datetime.now() # Pobieramy czas
    tx = {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'hour': czas.hour,
        'timestamp': czas.isoformat(), #znacznik czasowy - czas wygenerowania zdarzenia
    }

    # 2. dla 5% transakcji nadpisujemy pola tak, żeby spełniała warunki podejrzanej
    
    if random.random() < 0.05:
        tx['amount'] = round(random.uniform(3000.0, 5000.0), 2)
        tx['category'] = 'elektronika'
        tx['hour'] = random.randint(0, 5)
    return tx



for i in range(100):
    tx = generate_transaction()
    producer.send('koniec', value=tx) #koniec - nazwa topicu
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

Overwriting producer.py


In [11]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'koniec',
    bootstrap_servers='broker:9092', #serverS bo jednym konsumentem moge pobierac info z wielu brokerow
    #uto_offset_reset='earliest',
    #group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) #binarny kod na utf8
)

# TWÓJ KOD
# Dla każdej wiadomości: sprawdź amount > 1000, jeśli tak — wypisz ALERT
for message in consumer:
    wartosc_transakcji = message.value['amount']
    if wartosc_transakcji> 1000:
        print("%.2f: ALERT" % wartosc_transakcji)

Overwriting consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'koniec',
    bootstrap_servers='broker:9092', #serverS bo jednym konsumentem moge pobierac info z wielu brokerow
    #uto_offset_reset='earliest',
    #group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) #binarny kod na utf8
)

# TWÓJ KOD
# Napisz konsumenta, który dodaje pole risk_level: - amount > 3000 → “HIGH” - amount > 1000 → “MEDIUM” - reszta → “LOW”
for message in consumer:
    #wyciagniecie danych transakcji - slownik
    tx = message.value
    wartosc_transakcji = message.value['amount']
    
    if wartosc_transakcji > 3000:
        risk = 'HIGH'
    elif wartosc_transakcji > 1000:
        risk = 'MEDIUM'
    else:
        risk = 'LOW' 
    
    tx['risk_level'] = risk

    print(f"ID: {tx['tx_id']} | Kwota: {tx['amount']:>8.2f} | Ryzyko: {tx['risk_level']}")

Overwriting consumer_enrich.py


In [8]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'koniec',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. store_counts[store] += 1
#   2. total_amount[store] += amount
#   3. Co 10 wiadomości: print tabela

for message in consumer:
    tx = message.value
    store = tx['store']
    store_counts[store] += 1
    total_amount[store] += tx['amount']
    if msg_count % 10 == 0:  # co 10 wiadomości
        
        print(f"\n{'--- RAPORT SPRZEDAŻY':^45}") # ^ centruje tekst
            # Nagłówek tabeli: < oznacza wyrównanie do lewej, > do prawej, cyfra to szerokość
        print(f"{'Sklep':<15} | {'Transakcje':<12} | {'Suma (PLN)':>12}")
        print("-" * 45)
        
        for store in sorted(store_counts.keys()):
            count = store_counts[store]
            amount = total_amount[store]
            print(f"{store:<15} | {count:<12} | {amount:>12.2f}")
        
        print("-" * 45 + "\n")

    msg_count += 1



Overwriting consumer_count.py


In [3]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json
import time



def score_transaction(tx):
    score = 0
    rules = []
    # TWÓJ KOD — zaimplementuj reguły R1, R2, R3
    # R1: Wysoka kwota
    if tx['amount'] > 3000:
        score += 3
        rules.append("R1: Kwota > 3000")

    # R2: Elektronika i kwota>1500
    if tx['category'] == 'elektronika'and tx['amount'] > 1500:
        score += 2
        rules.append("R2: Elektronika i kwota > 1500")

    # R3: Godzina nocna 
    if 0 <= tx['hour'] <= 5:
        score += 2
        rules.append("R3: Godzina nocna")
    
    return score, rules

# Test
#test_tx = {'tx_id': 'TX999', 'amount': 4500.0, 'category': 'elektronika',
#           'timestamp': '2026-04-01T03:15:00', 'hour': 3}
#print(score_transaction(test_tx))  # powinno dać score >= 5


consumer = KafkaConsumer('koniec', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

for message in consumer:
    tx = message.value
    tx_score, rules = score_transaction(tx)
    
    if tx_score >= 3:
        alert_payload = {
            'tx_id': tx['tx_id'],
            'score': tx_score,
            'rules': rules,
            'original_tx': tx  # dołączamy dane transakcji dla analityka
        }
        alert_producer.send('alerts', value=alert_payload)
        print(f"UWAGA! Wysłano alert dla {tx['tx_id']} (Score: {tx_score})")

    time.sleep(1)
        

Writing scoring_consumer.py


In [4]:
%%file consumer_topicAlerts.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'alerts',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    print(message)

Writing consumer_topicAlerts.py


In [5]:
###